# Experimentos en Colab: CodeLlama-7B-Instruct a 16, 8 y 4 bits

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

Los resultados se guardan en Google Drive (`MyDrive/cuantizacion/resultados`). Si Colab se desconecta,
vuelve a ejecutar todas las celdas: el script salta los casos que ya están hechos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b experimentos/diseno https://github.com/CristhianBravo865/Articulo_Construccion_Software.git repo
!pip install -q -r repo/experimentos/requirements.txt

In [ ]:
# Resultados y datos persistentes en Drive
import os
DRIVE = '/content/drive/MyDrive/cuantizacion'
for sub in ['resultados', 'datos']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
    local = f'/content/repo/experimentos/{sub}'
    !rm -rf {local} && ln -s {DRIVE}/{sub} {local}
!ls -la /content/repo/experimentos

## Preparar los datos (solo la primera vez)
Descarga FixMe (815 MB, Zenodo) y SecurityEval, y genera `E1.jsonl` y `E2.jsonl`.
El embudo debe dar: E1_muestra 150 y E2_con_hallazgo_bandit 49 (igual que en la laptop).

In [ ]:
%cd /content/repo/experimentos/scripts
if not os.path.exists('../datos/E1.jsonl'):
    !wget -q -O /content/FixMe-v1.db "https://zenodo.org/records/10955342/files/FixMe-v1.db?download=1"
    !md5sum /content/FixMe-v1.db   # esperado: fd5de8fd5f9f85f380787a641b72182e
    !python 01_preparar_dataset.py --fixme /content/FixMe-v1.db
!cat ../datos/embudo.json

## Generar parches
Orden sugerido: primero E2 (49 casos, más corto), luego E1 (150 casos).
Prueba con `--limite 3` la primera vez para ver que todo funciona.

In [ ]:
%cd /content/repo/experimentos/scripts
for exp in ['E2', 'E1']:
    for cfg in ['L-16', 'L-8', 'L-4']:
        !python 02_generar_parches.py --experimento {exp} --config {cfg} --hardware colab-t4

## Medición controlada de costo (RQ3)
Mismos 10 prompts, 128 tokens exactos, 3 repeticiones por configuración (~10 min en total).

In [ ]:
%cd /content/repo/experimentos/scripts
for cfg in ['L-16', 'L-8', 'L-4']:
    !python 05_medir_costo.py --config {cfg} --hardware colab-t4

## Evaluar y resumir

In [ ]:
%cd /content/repo/experimentos/scripts
!python 03_evaluar.py ../resultados/gen_*_colab-t4.jsonl
!python 04_analizar.py